In [15]:
import gc
from pathlib import Path
import pandas as pd
import geopandas as gpd
import numpy as np
import pyreadstat
import matplotlib.pyplot as plt
import seaborn as sns

data_dir = Path(Path.cwd()).parent / "Data"
plot_dir = Path(Path.cwd()) / 'Plots'

In [ ]:
# Shapefiles
kreise = gpd.read_file(Path(data_dir / "Shapefiles-2022" / "Germany" / "COUNTIES" / "VG250_KRS_clean_final.shp"))

# Join county_id 16056 into county_id 16063
from shapely.ops import unary_union
kreise.loc[kreise["RS"].isin(["16056", "16063"]), "geometry"] = unary_union(
    kreise.loc[kreise["RS"].isin(["16056", "16063"]), "geometry"]
)
kreise = gpd.GeoDataFrame(kreise[~kreise["RS"].isin(["16056"])].copy())

In [17]:
# Translate the R block to pandas
labour_data = pd.read_excel(
    Path(data_dir / "county-wages-2022" / "country-wages-2022-entgelt-dwolk-0-202212-xlsx.xlsx"),
    sheet_name="8.1",
    usecols="A:K",
    skiprows=8,
    nrows=6287,
    dtype=str,
 )
# Stichtag 31.12.2024

# Rename and keep relevant variables (columns 1:4 and 11 in R's 1-based indexing)
labour_data = labour_data.iloc[:, [0, 1, 2, 3, 10]].copy()
labour_data.columns = [
    "AGS",      # Region key (Amtlicher Gemeindeschluessel)
    "region",
    "merkmale",
    "N",        # total workers
    "wages",    # median euro
 ]

# Select counties only
labour_data["AGS"] = labour_data["AGS"].astype("string").str.strip()
labour_data["RS"] = labour_data["AGS"]
labour_data = labour_data[labour_data["AGS"].str.len() == 5].copy()

# Select correct qualifications only
labour_data = labour_data[labour_data["merkmale"].isin([
    "ohne Berufsabschluss",
    "anerkannter Berufsabschluss",
    "akademischer Berufsabschluss",
    "Insgesamt",
])].copy()

labour_data.head()

,AGS,region,merkmale,N,wages,RS
286,01001,"Flensburg, Stadt",Insgesamt,25761,3433.1826196473553,01001
294,01001,"Flensburg, Stadt",ohne Berufsabschluss,2182,2395.6923076923076,01001
295,01001,"Flensburg, Stadt",anerkannter Berufsabschluss,16986,3409.590909090909,01001
296,01001,"Flensburg, Stadt",akademischer Berufsabschluss,4602,4891.728070175439,01001
301,01002,"Kiel, Landeshauptstadt",Insgesamt,77066,3772.814049586777,01002


In [18]:
housing_data = pd.read_csv("county_fixed_effects_demeaned.csv")

In [19]:
housing_data['county_id'] = housing_data['0'].astype(str).str.zfill(5)

In [20]:
data = labour_data
for id in data['AGS'].unique():
    data.loc[data['AGS'] == id, 'price_index'] = housing_data.loc[housing_data['county_id'] == id, 'purchase_price_index'].values[0]


In [21]:
# If X, then np.nan. If > or < then the number without the symbol.
# Numbers are in format e.g. 2621.298319327731 
def convert_wages(value):
    if value == 'X':
        return np.nan
    elif value.startswith('>'):
        return float(value[1:].strip().replace('.', ''))
    elif value.startswith('<'):
        return float(value[1:].strip().replace('.', ''))
    else:
        return float(value.strip())

v_convert_wages = np.vectorize(convert_wages)
data['wages'] = v_convert_wages(data['wages'])
data['N'] = data['N'].astype(float)

In [ ]:
# Set up
# Input share of labour
alpha = 2/3
# Expenditure share of housing
sigma = 1/3
# Height elasticity of construction cost
delta = 3/2

# beta = (1+ (1-alpha)*b_w^*)/(1+b_w^*) <- to be estimated in the next step

In [ ]:
skill_groups = data['merkmale'].unique()
# Create logs of wage and labour supply
data['log_wages'] = np.log(data['wages'])
data['log_N'] = np.log(data['N'])

import statsmodels.formula.api as smf

b_w_stars = {}
for skill in skill_groups:
    formula = f"log_wages ~ log_N"
    model = smf.ols(formula=formula, data=data[data['merkmale'] == skill]).fit()
    b_w_stars[skill] = model.params['log_N']

In [ ]:
cobb_douglas_params = {'alpha': alpha, 'sigma': sigma, 'delta': delta, 'beta': {skill: (1 + (1 - alpha) * b_w_stars[skill]) / (1 + b_w_stars[skill]) for skill in skill_groups}}
data['beta'] = data['merkmale'].map(cobb_douglas_params['beta'])

In [ ]:
data['qol'] = -data['log_wages'] + sigma * data['price_index']
data['fund_prod'] = (1-data['beta']) * data['log_wages'] + (1-cobb_douglas_params['alpha'] - data['beta']) * data['log_N']
data['eff_land_supply'] = data['log_wages'] + data['log_N'] - (cobb_douglas_params['delta']/(cobb_douglas_params['delta']-1)) * data['price_index']


In [ ]:
# Set up for plotting
# Params for clean and minimalistic plots
sns.set_style("whitegrid")
plt.rcParams.update({
    'figure.figsize': (6*2, 2*4.5),
    'font.size': 14.0,
    'font.family': 'serif',
    'font.serif': 'Palatino',
    'axes.titlesize': 'medium',
    'figure.titlesize': 'large',
    'legend.fontsize': 'medium',
    # dpi for high-res output
    'figure.dpi': 100,
    'savefig.dpi': 300,
    # Tight layout by default
    'figure.autolayout': True,
    'text.usetex': True,
    'text.latex.preamble': r"\usepackage{amsmath}\usepackage{amssymb}\usepackage{siunitx}[=v2]",
})

286    -7.833832
294    -7.474017
295    -7.826937
296    -8.187890
301    -7.870722
          ...   
6266   -8.349423
6271   -7.860055
6279   -7.691795
6280   -7.844686
6281   -8.312958
Name: qol, Length: 1600, dtype: float64

In [ ]:
# Plotting on a map
    